Notebook to test on real data

In [1]:
import pandas as pd

from datkit.cleaning import clean_dataframe

Import table

In [3]:
# don't use dtype=str as that creates legacy objects that pythins stores strings in
# pyarrow is a better option for string storage and manipulation
df = pd.read_csv(
    "d:/dev/data-analysis-toolkit/devdata/title.basics.tsv.gz", sep="\t", dtype_backend="pyarrow", nrows=100_000
)

clean table

In [7]:
df.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
43660,tt0044455,movie,Buffalo Bill in Tomahawk Territory,Buffalo Bill in Tomahawk Territory,0,1952,\N,66,"Drama,Western"
87278,tt0089267,movie,Heilende Schläge,Heilende Schläge,0,1985,\N,\N,\N
14317,tt0014555,movie,Toilers of the Sea,Toilers of the Sea,0,1923,\N,60,Drama
81932,tt0083769,movie,Sha ren ai qing jie,Sha ren ai qing jie,0,1982,\N,90,"Action,Romance"
95321,tt0097513,movie,Hisaab Khoon Ka,Hisaab Khoon Ka,0,1989,\N,\N,"Drama,Mystery"


In [5]:
df_clean, cleaning_report = clean_dataframe(df, False)

D:\dev\data-analysis-toolkit\src\datkit\cleaning.py:122: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sample_dt_dayftrue = pd.to_datetime(dt, dayfirst=True, errors="coerce")
D:\dev\data-analysis-toolkit\src\datkit\cleaning.py:123: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sample_dt_dayffalse = pd.to_datetime(dt, dayfirst=False, errors="coerce")
D:\dev\data-analysis-toolkit\src\datkit\cleaning.py:122: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sample_dt_dayftrue = pd.to_datetime(dt, dayfirst=True, errors="coerce")
D:\dev\data-analysis-toolkit\src\datkit\cleaning.py:1

In [8]:
df_clean.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
43660,tt0044455,movie,Buffalo Bill in Tomahawk Territory,Buffalo Bill in Tomahawk Territory,0,1952,<NA>,66,"Drama,Western"
87278,tt0089267,movie,Heilende Schläge,Heilende Schläge,0,1985,<NA>,<NA>,<NA>
14317,tt0014555,movie,Toilers of the Sea,Toilers of the Sea,0,1923,<NA>,60,Drama
81932,tt0083769,movie,Sha ren ai qing jie,Sha ren ai qing jie,0,1982,<NA>,90,"Action,Romance"
95321,tt0097513,movie,Hisaab Khoon Ka,Hisaab Khoon Ka,0,1989,<NA>,<NA>,"Drama,Mystery"


In [9]:
cleaning_report

,column,dtype_before,dtype_after,converted,bytes_before,bytes_after,bytes_saved,mb_before,mb_after,mb_saved,nulls_before,nulls_after,nulls_created
0,tconst,string[pyarrow],string[pyarrow],False,1300000,1312500,-12500,1.24,1.25,-0.01,0,0,0
1,titleType,string[pyarrow],string[pyarrow],False,936793,949293,-12500,0.89,0.91,-0.01,0,0,0
2,primaryTitle,string[pyarrow],string[pyarrow],False,2139127,2151627,-12500,2.04,2.05,-0.01,0,0,0
3,originalTitle,string[pyarrow],string[pyarrow],False,2155967,2168467,-12500,2.06,2.07,-0.01,0,0,0
4,isAdult,int64[pyarrow],int64[pyarrow],False,800000,800000,0,0.76,0.76,0.00,0,0,0
5,startYear,string[pyarrow],int64[pyarrow],True,799974,812500,-12526,0.76,0.77,-0.01,0,13,13
6,endYear,string[pyarrow],int64[pyarrow],True,608384,812500,-204116,0.58,0.77,-0.19,0,95808,95808
7,runtimeMinutes,string[pyarrow],int64[pyarrow],True,613511,812500,-198989,0.59,0.77,-0.19,0,12376,12376
8,genres,string[pyarrow],string[pyarrow],False,1607178,1608584,-1406,1.53,1.53,-0.00,0,5547,5547


In [11]:
def sample_records(df: pd.DataFrame, n=10000, seed=1) -> pd.DataFrame:
    """Return a sample of records from the DataFrame."""
    if len(df) > n:
        return df.sample(n=n, random_state=seed)
    else:
        return df

In [12]:
df_clean_sample = sample_records(df_clean)

In [14]:
df_sample_nonunique = df_clean_sample.nunique()

In [15]:
df_sample_nonunique

tconst            10000
titleType            10
primaryTitle       9835
originalTitle      9874
isAdult               2
startYear           108
endYear              55
runtimeMinutes      252
genres              576
dtype: int64

In [18]:
df_sample_nonunique["isAdult"]

np.int64(2)

for col in df_clean_sample.columns:
    if df_sample_nonunique[col] > 100:
        if "string" in cleaning_report[cleaning_report["column"] == col]["dtype_after"].values[0]:
            pass
        else:
            df_clean_sample[col] = pd.qcut(df_clean_sample[col], q=4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
    elif df_sample_nonunique[col] > 10:
    else:

In [21]:
print(cleaning_report[cleaning_report["column"] == "isAdult"]["dtype_after"].values[0])

int64[pyarrow]


In [32]:
df_clean_sample["runtimeMinutes_bins"] = pd.cut(df_clean_sample["runtimeMinutes"], bins=5, precision=0)

In [33]:
df_clean_sample.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,runtimeMinutes_Quartiles,runtimeMinutes_bins
62709,tt0064002,movie,Alice's Restaurant,Alice's Restaurant,0,1969,<NA>,111,"Comedy,Drama,Music","(-0.427, 286.4]","(-0.0, 286.0]"
54703,tt0055777,movie,The Bashful Elephant,The Bashful Elephant,0,1962,<NA>,82,"Adventure,Family","(-0.427, 286.4]","(-0.0, 286.0]"
48716,tt0049613,movie,No Ordinary Summer,Pervye radosti,0,1957,<NA>,103,Drama,"(-0.427, 286.4]","(-0.0, 286.0]"
74446,tt0076060,movie,Funeral for an Assassin,Funeral for an Assassin,0,1974,<NA>,92,"Crime,Drama","(-0.427, 286.4]","(-0.0, 286.0]"
60846,tt0062073,movie,Date for a Murder,Omicidio per appuntamento,0,1967,<NA>,105,"Action,Crime,Thriller","(-0.427, 286.4]","(-0.0, 286.0]"


In [36]:
df_clean_sample.groupby("runtimeMinutes_bins")["runtimeMinutes"].agg(["count", "min", "max", "mean", "median", "std"])

C:\Users\debee\AppData\Local\Temp\ipykernel_15116\20456517.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_clean_sample.groupby("runtimeMinutes_bins")["runtimeMinutes"].agg(["count", "min", "max", "mean", "median", "std"])


,count,min,max,mean,median,std
runtimeMinutes_bins,,,,,,
"(-0.0, 286.0]",8727,1,282,76.366105,84.0,34.945298
"(286.0, 572.0]",36,288,540,350.305556,336.5,63.759288
"(572.0, 857.0]",2,624,763,693.5,693.5,98.287843
"(857.0, 1143.0]",0,<NA>,<NA>,<NA>,<NA>,<NA>
"(1143.0, 1428.0]",1,1428,1428,1428.0,1428.0,<NA>
